# International Day Pass — Synthetic Data Generation

**Purpose.** This notebook builds a synthetic dataset for the day pass pricing analysis. No AT&T data is used, read, or reproduced anywhere in this project. Every value here is generated from parameters chosen by hand to be plausible.

**Why synthesize rather than mock.** A random table would let the modeling code *run*; it would not let it be *checked*. This notebook generates data from an explicit behavioral process, so the true price sensitivity of every account is known. When the multiclass model is fit later, its estimates can be compared against the values that produced the data — which is a stronger test than anything real data offers.

**What comes out**

| Table | Grain | Used for |
|---|---|---|
| `accounts` | one row per account | features, segmentation |
| `macro` | one row per month | simulation scenarios |
| `trips` | one row per account-trip | the modeling unit |
| `outcomes` | one row per account-trip | labels and revenue |
| `ground_truth` | one row per account | validation only — never a model input |

**Where randomness enters.** Deliberately at four levels, because a dataset with noise in only one place produces models that look far better than they are:

1. **Account heterogeneity** — each account draws its own latent price sensitivity and travel propensity.
2. **Macro path** — an AR(1) economic index shared by everyone, so trip volumes are *correlated across accounts*. This is the structure the plan flags as easy to get wrong in simulation.
3. **Trip-level draws** — trip counts, party size, and duration are all random given the account.
4. **Choice noise** — the behavioral state is drawn from a probability vector, not assigned deterministically.

## Plan

The generator runs in seven steps, each depending only on the ones above it.

1. **Accounts** — structure (lines, tenure, plan, business flag) and two latent traits: travel propensity and price sensitivity.
2. **Macro path** — 24 months of an AR(1) economic index; converted to a travel-volume multiplier applied to everyone at once.
3. **Trips** — Poisson draws per account-month, scaled by propensity × seasonality × macro. Each trip gets a destination, a party size, and a duration.
4. **Price faced** — the historical price variation the model will eventually learn from. Without variation there is nothing to estimate, so this is generated explicitly rather than assumed.
5. **Behavioral state** — a multinomial logit over the five states, driven by the account's latent sensitivity and the price it faced.
6. **Magnitude** — for the two states that need sizing (reduce, increase), how many lines and days were actually bought.
7. **Revenue** — the pricing identity applied to the realized lines and days.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RNG = np.random.default_rng(20260820)
OUT = Path("data/synthetic")
OUT.mkdir(parents=True, exist_ok=True)

N_ACCOUNTS = 50_000
N_MONTHS = 24

# Product pricing
PRICE_FIRST_LINE = 10.0
PRICE_ADDL_LINE = 5.0

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

## 1 · Accounts

Two latent traits matter and neither is directly observable in real data:

- **`travel_propensity`** — expected international trips per month, lognormal so a minority of accounts travel far more than the median.
- **`price_sensitivity`** — how strongly this account reacts to a price change. Negative; nearer zero means less sensitive.

Sensitivity is generated as a function of observable characteristics *plus* an unexplained residual. That residual is the point: it means the eventual model cannot recover sensitivity perfectly, which is exactly the situation in production. A simulation where features fully determine the outcome teaches nothing about how the model will behave later.

In [2]:
def make_accounts(n, rng):
    is_business = rng.random(n) < 0.22

    # Business accounts skew to more lines
    lines = np.where(
        is_business,
        rng.choice([1, 2, 3, 4, 5], size=n, p=[0.15, 0.20, 0.25, 0.22, 0.18]),
        rng.choice([1, 2, 3, 4, 5], size=n, p=[0.38, 0.31, 0.17, 0.09, 0.05]),
    )

    tenure_months = rng.integers(1, 145, size=n)
    plan_tier = rng.choice(["value", "standard", "premium"], size=n, p=[0.30, 0.45, 0.25])
    premium = (plan_tier == "premium").astype(float)

    # Travel propensity: trips per month, lognormal
    log_prop = (
        -3.60
        + 1.55 * is_business
        + 0.35 * premium
        + 0.004 * (tenure_months - 72) / 12
        + rng.normal(0, 0.95, size=n)
    )
    travel_propensity = np.exp(log_prop)

    # Latent price sensitivity: negative, nearer zero = less sensitive.
    # Reimbursed business travel and premium plans are less sensitive;
    # the residual keeps it from being a deterministic function of features.
    price_sensitivity = (
        -1.60
        + 0.95 * is_business
        + 0.30 * premium
        + 0.08 * np.log1p(travel_propensity * 12)
        + rng.normal(0, 0.40, size=n)
    )
    price_sensitivity = np.clip(price_sensitivity, -3.0, -0.05)

    return pd.DataFrame({
        "account_id": np.arange(n),
        "is_business": is_business,
        "lines_on_account": lines,
        "tenure_months": tenure_months,
        "plan_tier": plan_tier,
        "travel_propensity": travel_propensity,
        "price_sensitivity": price_sensitivity,
    })


accounts = make_accounts(N_ACCOUNTS, RNG)
accounts.head()

,account_id,is_business,lines_on_account,tenure_months,plan_tier,travel_propensity,price_sensitivity
0,0,False,2,72,premium,0.008166,-0.817886
1,1,True,5,76,premium,0.615436,-0.265488
2,2,False,2,117,value,0.027803,-1.333017
3,3,False,2,97,standard,0.022454,-1.928163
4,4,True,1,2,premium,0.144453,-0.819382


## 2 · Macro path

An AR(1) index standing in for economic expansion and contraction, converted to a multiplier on travel volume.

This is the piece that makes the later Monte Carlo honest. Every account's trip count in a given month is scaled by the *same* draw, so account outcomes are correlated. Simulating accounts independently would shrink the revenue distribution and report confidence the analysis has not earned.

In [3]:
def make_macro(n_months, rng, phi=0.85, sigma=0.35, gamma=0.55):
    """AR(1) economic index -> travel volume multiplier.

    phi   persistence; gamma how strongly the economy moves travel.
    """
    idx = np.zeros(n_months)
    for t in range(1, n_months):
        idx[t] = phi * idx[t - 1] + rng.normal(0, sigma)

    month = np.arange(n_months)
    # Northern-hemisphere leisure travel peaks mid-year and in December
    seasonality = (
        1.0
        + 0.15 * np.sin(2 * np.pi * (month % 12 - 2) / 12)
        + 0.06 * ((month % 12) == 11)
    )

    return pd.DataFrame({
        "month": month,
        "macro_index": idx,
        "seasonality": seasonality,
        "travel_multiplier": np.exp(gamma * idx) * seasonality,
    })


macro = make_macro(N_MONTHS, RNG)
macro.head()

,month,macro_index,seasonality,travel_multiplier
0,0,0.000000,0.870096,0.870096
1,1,-0.035610,0.925000,0.907060
2,2,-0.475274,1.000000,0.769973
3,3,-0.500878,1.075000,0.816146
4,4,-0.701075,1.129904,0.768389


## 3 · Trips

Poisson trip counts per account-month. Each trip carries a destination region, how many of the account's lines travelled, and how many days.

Destination matters beyond geography: it proxies how easy the substitute is. A traveller in a market with mature eSIM coverage has a real alternative to the pass, which caps how far price can move regardless of what the elasticity says.

In [4]:
REGIONS = ["europe", "latam", "apac", "canada", "other"]
REGION_P = [0.34, 0.22, 0.18, 0.16, 0.10]
# Higher = easier to substitute the pass with eSIM / local SIM / Wi-Fi
SUBSTITUTE_EASE = {"europe": 0.85, "latam": 0.55, "apac": 0.70, "canada": 0.40, "other": 0.45}


def make_trips(accounts, macro, rng):
    prop = accounts["travel_propensity"].to_numpy()[:, None]      # (n, 1)
    mult = macro["travel_multiplier"].to_numpy()[None, :]         # (1, m)

    counts = rng.poisson(prop * mult)                             # (n, m)
    acct_idx, month_idx = np.nonzero(counts)
    reps = counts[acct_idx, month_idx]

    account_id = np.repeat(accounts["account_id"].to_numpy()[acct_idx], reps)
    month = np.repeat(month_idx, reps)
    n_trips = len(account_id)

    lines_available = accounts.set_index("account_id").loc[account_id, "lines_on_account"].to_numpy()
    is_business = accounts.set_index("account_id").loc[account_id, "is_business"].to_numpy()

    # Business trips are usually solo; leisure trips bring the household
    solo_p = np.where(is_business, 0.82, 0.32)
    solo = rng.random(n_trips) < solo_p
    lines_traveling = np.where(
        solo,
        1,
        1 + rng.binomial(np.maximum(lines_available - 1, 0), 0.75),
    )

    region = rng.choice(REGIONS, size=n_trips, p=REGION_P)

    # Duration: business trips shorter and tighter, leisure longer and more variable
    days = np.where(
        is_business,
        rng.lognormal(1.35, 0.40, size=n_trips),
        rng.lognormal(1.80, 0.60, size=n_trips),
    )
    days_traveled = np.clip(np.round(days), 1, 30).astype(int)

    return pd.DataFrame({
        "trip_id": np.arange(n_trips),
        "account_id": account_id,
        "month": month,
        "region": region,
        "lines_traveling": lines_traveling.astype(int),
        "days_traveled": days_traveled,
        "substitute_ease": [SUBSTITUTE_EASE[r] for r in region],
    })


trips = make_trips(accounts, macro, RNG)
print(f"{len(trips):,} trips across {trips.account_id.nunique():,} accounts")
trips.head()

94,440 trips across 29,312 accounts


,trip_id,account_id,month,region,lines_traveling,days_traveled,substitute_ease
0,0,1,0,canada,1,4,0.40
1,1,1,5,latam,1,5,0.55
2,2,1,5,latam,1,4,0.55
3,3,1,8,canada,5,4,0.40
4,4,1,8,europe,1,2,0.85


## 4 · Price faced

The most important step for realism, and the one most likely to be missing from real data.

A price response can only be estimated where the price actually moved. Here that variation is generated three ways: a market-level test that ran in some regions, a promotional window, and a small amount of account-level pricing drift. Roughly a third of trips end up facing something other than the standard rate.

The multiplier is deliberately **correlated with region and period, not random** — which reproduces the confounding problem the real analysis faces. Expensive markets and heavy travellers overlap, so a naive fit will attribute those differences to price. That is a feature of this dataset, not a flaw: it lets the modeling notebook demonstrate handling it.

In [5]:
def assign_price(trips, rng):
    n = len(trips)
    mult = np.ones(n)

    # (a) Market test: two regions carried a higher rate for months 8-15
    in_test = trips["region"].isin(["europe", "apac"]).to_numpy() & trips["month"].between(8, 15).to_numpy()
    mult = np.where(in_test, rng.choice([1.10, 1.20], size=n, p=[0.5, 0.5]), mult)

    # (b) Promotional window: a discount in the shoulder season
    in_promo = trips["month"].isin([4, 5, 16, 17]).to_numpy() & (rng.random(n) < 0.55)
    mult = np.where(in_promo & ~in_test, rng.choice([0.90, 0.95], size=n, p=[0.6, 0.4]), mult)

    # (c) Residual drift so the variation is not purely two clean blocks
    drift = (rng.random(n) < 0.10)
    mult = np.where(drift & ~in_test & ~in_promo, rng.choice([0.95, 1.10], size=n), mult)

    out = trips.copy()
    out["price_multiplier"] = mult
    out["price_first_line"] = PRICE_FIRST_LINE * mult
    out["price_addl_line"] = PRICE_ADDL_LINE * mult
    return out


trips = assign_price(trips, RNG)
print(trips["price_multiplier"].value_counts(normalize=True).sort_index().round(3))

price_multiplier
0.90    0.065
0.95    0.082
1.00    0.671
1.10    0.109
1.20    0.073
Name: proportion, dtype: float64


## 5 · Behavioral state

A multinomial logit over the five states, with `unchanged` as the reference. Utilities move with `sensitivity × log(price multiplier)`, so a price rise pushes probability toward leaving, substituting and reducing, and a price cut pushes it toward increasing.

Three structural choices worth naming, because they are what make the dataset hard in the right ways:

- **Leaving is rare** (a large negative intercept) but expensive. This reproduces the class-imbalance problem the plan flags: an untreated model will barely predict it and the simulation will understate the downside.
- **Substitution scales with `substitute_ease`.** Where a good alternative exists, the price ceiling is lower — regardless of how price-insensitive the traveller is.
- **Increasing is only reachable on a discount.** Nobody buys more days because the price went up.

In [6]:
STATES = ["leaves", "substitutes", "reduces", "unchanged", "increases"]


def state_probabilities(trips, accounts):
    a = accounts.set_index("account_id").loc[trips["account_id"].to_numpy()]
    sens = a["price_sensitivity"].to_numpy()          # negative
    is_business = a["is_business"].to_numpy()
    tenure = a["tenure_months"].to_numpy()

    logp = np.log(trips["price_multiplier"].to_numpy())   # >0 increase, <0 discount
    ease = trips["substitute_ease"].to_numpy()
    party = trips["lines_traveling"].to_numpy()

    # Relief from the multi-line rate: a solo traveler pays the full first-line
    # rate per head; a party of `party` pays (first + addl*(party-1))/party per
    # head, which falls below the first-line rate as party grows. Expressed as
    # a 0-to-~0.4 "relief" score (0 = solo, no relief) so that larger, more
    # relieved parties get a MORE negative u_reduces term below — i.e. they are
    # structurally less exposed to a headline increase and less likely to cut
    # back. (Earlier version of this ratio ran the other way and had larger
    # parties reducing *more* than solo travelers — the opposite of the
    # intended relationship.)
    eff_price_relief = 1.0 - (PRICE_FIRST_LINE + PRICE_ADDL_LINE * (party - 1)) / (PRICE_FIRST_LINE * party)

    u_unchanged = np.zeros(len(trips))

    u_leaves = (
        -5.20
        + (-sens) * 1.10 * logp
        - 0.55 * is_business
        - 0.004 * tenure
    )

    u_substitutes = (
        -2.60
        + (-sens) * 1.85 * logp
        + 1.30 * ease
        - 0.70 * is_business
    )

    u_reduces = (
        -1.90
        + (-sens) * 1.55 * logp
        + 0.45 * ease
        - 0.40 * is_business
        - 0.80 * eff_price_relief
    )

    # Only reachable when logp < 0
    u_increases = (
        -2.40
        + (-sens) * 2.10 * (-np.minimum(logp, 0))
        - 3.00 * np.maximum(logp, 0)
    )

    U = np.column_stack([u_leaves, u_substitutes, u_reduces, u_unchanged, u_increases])
    U = U - U.max(axis=1, keepdims=True)
    P = np.exp(U)
    return P / P.sum(axis=1, keepdims=True)


P = state_probabilities(trips, accounts)
drawn = np.array([RNG.choice(5, p=row) for row in P]) if len(P) < 5_000 else (
    (P.cumsum(axis=1) < RNG.random((len(P), 1))).sum(axis=1)
)
trips["state"] = [STATES[i] for i in drawn]

print(trips["state"].value_counts(normalize=True).round(4))

state
unchanged      0.7281
reduces        0.1128
substitutes    0.0924
increases      0.0643
leaves         0.0024
Name: proportion, dtype: float64


## 6 · Magnitude

Three states are self-sizing. `unchanged` buys every travel day for every travelling line; `substitutes` and `leaves` buy nothing.

The other two need a size, and the size responds to price: a bigger increase produces a deeper cut, not just more cutters. Both the day count and the line count can move, since a household can put one phone on the pass and share it.

**Unchanged is not exactly full coverage.** A minority of unchanged trips buy one day less than they travelled for reasons that have nothing to do with price: a late arrival, a leg with hotel wifi, forgetting to activate on the last day. This incidental shortfall is generated deliberately. Without it, every unchanged trip sits exactly at 1.0, the reduce-versus-unchanged boundary is trivially separable, and the labelling notebook cannot test a decision that in production is genuinely hard.

In [7]:
def make_magnitudes(trips, accounts, rng):
    n = len(trips)
    state = trips["state"].to_numpy()
    days_t = trips["days_traveled"].to_numpy()
    lines_t = trips["lines_traveling"].to_numpy()
    logp = np.log(trips["price_multiplier"].to_numpy())
    # Physical cap on how many lines an account can activate — needed below so
    # "increases" trips can't add a line past what the account actually has.
    lines_cap = accounts.set_index("account_id").loc[trips["account_id"].to_numpy(), "lines_on_account"].to_numpy()

    days_bought = days_t.copy().astype(float)
    lines_active = lines_t.copy().astype(float)

    # --- reduces: keep the pass, buy less of it ---
    m = state == "reduces"
    # Deeper cuts at higher prices: Beta mean shifts down as logp rises
    mean_share = np.clip(0.55 - 1.20 * logp[m], 0.10, 0.85)
    conc = 6.0
    share = rng.beta(mean_share * conc, (1 - mean_share) * conc)
    days_bought[m] = np.maximum(np.round(days_t[m] * share), 1)
    # Some of the reduction comes out of lines instead of days
    drop_line = (rng.random(m.sum()) < 0.35) & (lines_t[m] > 1)
    la = lines_active[m]
    la[drop_line] = np.maximum(np.round(la[drop_line] * rng.uniform(0.4, 0.8, drop_line.sum())), 1)
    lines_active[m] = la

    # --- increases: buy more days, occasionally add a line ---
    m = state == "increases"
    # Floor at 1: an account that responded to a discount must be observably different
    # Floor at 1 extra day, but let the DEPTH of the discount matter: a low clip
    # floor would pin the mean across the whole tested range and make increase
    # size price-invariant by construction.
    extra = 1 + rng.poisson(np.clip(-12.0 * logp[m], 0.05, 4.0))
    days_bought[m] = days_t[m] + extra
    add_line = (rng.random(m.sum()) < 0.25)
    la = lines_active[m]
    cap = lines_cap[m]
    # Cap at the account's total lines — can't activate more pass lines than exist.
    la[add_line] = np.minimum(la[add_line] + 1, cap[add_line])
    lines_active[m] = la

    # --- unchanged: incidental shortfall, not a price response ---
    # Arrival late in the day, a leg with wifi, forgetting to activate. This is the
    # noise that makes reduce-vs-unchanged a real decision rather than a formality.
    m = state == "unchanged"
    incidental = (rng.random(m.sum()) < 0.12) & (days_t[m] >= 3)
    db = days_bought[m]
    # Mostly one day, occasionally two: a rule that keys on "two or more days short"
    # must be able to misfire, or its precision is an artefact of the generator.
    lost = np.where(rng.random(incidental.sum()) < 0.20, 2, 1)
    db[incidental] = np.maximum(db[incidental] - lost, 1)
    days_bought[m] = db

    # --- no purchase ---
    m = np.isin(state, ["substitutes", "leaves"])
    days_bought[m] = 0
    lines_active[m] = 0

    out = trips.copy()
    out["days_bought"] = days_bought.astype(int)
    out["lines_active"] = lines_active.astype(int)
    return out


trips = make_magnitudes(trips, accounts, RNG)
trips.groupby("state")[["days_traveled", "days_bought", "lines_traveling", "lines_active"]].mean().round(2)

,days_traveled,days_bought,lines_traveling,lines_active
state,,,,
increases,5.40,6.60,1.41,1.56
leaves,6.23,0.00,1.42,0.00
reduces,5.73,3.04,1.36,1.27
substitutes,5.88,0.00,1.46,0.00
unchanged,5.42,5.29,1.41,1.41


## 7 · Revenue

The pricing identity applied to what was actually bought:

```
revenue = (price_first_line + price_addl_line × (lines_active − 1)) × days_bought
```

`leaves` is recorded separately. Losing the account costs far more than the pass — it removes the whole relationship — so carrying it as a zero-revenue trip would understate the damage. The account-level loss is attached here and left for the revenue model to weight, rather than being buried inside a trip row.

In [8]:
def compute_revenue(trips):
    out = trips.copy()
    active = out["lines_active"].to_numpy()
    rev = np.where(
        active > 0,
        (out["price_first_line"].to_numpy() + out["price_addl_line"].to_numpy() * (active - 1))
        * out["days_bought"].to_numpy(),
        0.0,
    )
    out["pass_revenue"] = rev
    out["account_lost"] = (out["state"] == "leaves").astype(int)
    return out


trips = compute_revenue(trips)

print(f"Total pass revenue: ${trips.pass_revenue.sum():,.0f}")
print(f"Accounts lost:      {trips.account_lost.sum():,}")
trips.groupby("state")["pass_revenue"].agg(["count", "sum", "mean"]).round(2)

Total pass revenue: $5,407,880
Accounts lost:      224


,count,sum,mean
state,,,
increases,6072,519138.0,85.50
leaves,224,0.0,0.00
reduces,10654,376216.0,35.31
substitutes,8730,0.0,0.00
unchanged,68760,4512525.5,65.63


## 8 · Sanity checks

A synthetic dataset is only useful if it behaves the way the domain says it should. If any of these fail, the parameters are wrong and nothing downstream is worth fitting.

1. Revenue per trip moves coherently with price, and the raw and composition-adjusted views agree.
2. Business accounts are measurably less sensitive than consumer accounts.
3. Substitution is more common where the alternative is easier.
4. Trip volume tracks the macro index.

In [9]:
chk = trips.merge(accounts[["account_id", "is_business"]], on="account_id")

print("1 · Revenue per trip by price multiplier — raw, then within segment x region")
raw = chk.groupby("price_multiplier")["pass_revenue"].mean().round(2)
adj = (chk.groupby(["is_business", "region", "price_multiplier"])["pass_revenue"].mean()
       .groupby("price_multiplier").mean().round(2))
print(pd.DataFrame({"raw": raw, "within segment x region": adj}).to_string(), "\n")
print("Expectation: revenue per trip RISES with price if demand is net inelastic in")
print("this range, which is the finding the project is testing for. What must not")
print("happen is the raw and adjusted columns telling different stories — that would")
print("mean the pattern is composition, not response.\n")

print("2 · Share of trips buying nothing, by price and segment")
chk["no_purchase"] = chk["days_bought"].eq(0)
print(chk.pivot_table(index="price_multiplier", columns="is_business",
                      values="no_purchase", aggfunc="mean").round(3), "\n")

print("3 · Substitution rate by region")
print(chk.assign(sub=chk.state.eq("substitutes")).groupby("region")["sub"].mean().round(3), "\n")

print("4 · Correlation of monthly trip volume with macro index")
vol = trips.groupby("month").size().rename("trips").reset_index().merge(macro, on="month")
print(round(vol["trips"].corr(vol["macro_index"]), 3))

1 · Revenue per trip by price multiplier — raw, then within segment x region
                    raw  within segment x region
price_multiplier                                
0.90              55.27                    57.79
0.95              55.66                    58.15
1.00              57.18                    59.98
1.10              58.43                    62.03
1.20              59.87                    62.60 

Expectation: revenue per trip RISES with price if demand is net inelastic in
this range, which is the finding the project is testing for. What must not
happen is the raw and adjusted columns telling different stories — that would
mean the pattern is composition, not response.

2 · Share of trips buying nothing, by price and segment
is_business       False  True 
price_multiplier              
0.90              0.091  0.059
0.95              0.116  0.063
1.00              0.118  0.067
1.10              0.165  0.080
1.20              0.205  0.100 

3 · Substitution rate by 

## 9 · Ground truth

Written to a separate file and **never joined into the modeling data**.

This is the advantage synthetic data has over the real thing: the modeling notebook can be scored on whether it recovers the sensitivity that actually generated the behavior, not just on how well it fits. If the model cannot recover a signal that is provably present, the problem is the model — not the data.

In [10]:
ground_truth = accounts[["account_id", "price_sensitivity", "travel_propensity"]].copy()
ground_truth["sensitivity_decile"] = pd.qcut(ground_truth["price_sensitivity"], 10, labels=False)

# Model-facing tables exclude the latent traits
model_accounts = accounts.drop(columns=["price_sensitivity", "travel_propensity"])

# Parquet is preferred (smaller, keeps dtypes) but needs pyarrow or fastparquet.
# Fall back to CSV so the notebook runs on a bare pandas install.
try:
    import pyarrow  # noqa: F401
    FMT = "parquet"
except ImportError:
    try:
        import fastparquet  # noqa: F401
        FMT = "parquet"
    except ImportError:
        FMT = "csv"
        print("pyarrow/fastparquet not found, writing CSV instead.")
        print("For smaller files:  pip install pyarrow\n")


def save(df, name):
    path = OUT / f"{name}.{FMT}"
    df.to_parquet(path, index=False) if FMT == "parquet" else df.to_csv(path, index=False)
    return path


for df, name in [(model_accounts, "accounts"), (macro, "macro"),
                 (trips, "trips"), (ground_truth, "ground_truth")]:
    save(df, name)

print(f"Written to {OUT.resolve()}")
for f in sorted(OUT.glob(f"*.{FMT}")):
    print(f"  {f.name:24s} {f.stat().st_size/1e6:6.2f} MB")

Written to /Users/paforsey/Inbox/Projects/data-science-portfolio/case-studies/international-roaming/research_notebooks/data/synthetic
  account_departure_risk.parquet   0.24 MB
  account_price_response.parquet   1.19 MB
  accounts.parquet           0.39 MB
  deterministic_revenue_curve.parquet   0.00 MB
  ground_truth.parquet       1.32 MB
  macro.parquet              0.00 MB
  magnitude_meta.parquet     0.00 MB
  model_meta.parquet         0.00 MB
  price_sweep.parquet        0.00 MB
  revenue_by_state.parquet  10.35 MB
  scored_trips.parquet       1.87 MB
  simulation_assumptions.parquet   0.00 MB
  simulation_price_sweep.parquet   0.01 MB
  simulation_scenarios.parquet   0.00 MB
  trip_magnitudes.parquet    1.79 MB
  trips.parquet              1.41 MB


---

## What this notebook deliberately does not do

**It does not assign labels.** `state` here is the *true* generating state. In the modeling notebook the labels have to be reconstructed from observable quantities — `days_bought` against `days_traveled`, `lines_active` against `lines_traveling` — exactly as they would be from real data. Comparing reconstructed labels against the true state is the cleanest possible test of the labeling rule, and it is the decision the plan flags as the one most likely to fail silently.

**It does not simulate forward.** This is the historical panel the models are fit on. The 12-month forward Monte Carlo is a separate notebook: it re-runs the fitted models across the −10% to +20% price range under expansion, base and contraction paths, drawing the macro shock **once per iteration and applying it to every account**, so revenue outcomes stay correlated.

**It does not tune anything to a target.** No parameter here was chosen to make the answer come out at a particular number. The generator is a plausible world; whatever the analysis finds in it is what it finds.

### Next

1. `02_label_construction.ipynb` — reconstruct the five states from observables; score the rule against `state`.
2. `03_state_model.ipynb` — multiclass model with temporal split, class weighting, calibration curves.
3. `04_magnitude_model.ipynb` — conditional day and line models for reduce and increase.
4. `05_simulation.ipynb` — 12-month Monte Carlo across the price range with correlated macro draws.
5. `06_segmentation.ipynb` — cluster on the probability vector, profile the segments, export to Power BI.